In [32]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path(r"C:\f1 new\data")

df = pd.read_parquet(OUTPUT_DIR / "f1_laps_silver.parquet")
print(f"Silver: {len(df):,} rows")

Silver: 94,777 rows


In [33]:
COMPOUND_MAX_LIFE = {
    'SOFT': 25, 'MEDIUM': 35,
    'HARD': 45, 'INTERMEDIATE': 30, 'WET': 40
}

df['compound_max_life'] = df['Compound'].map(COMPOUND_MAX_LIFE) 
df['TyreAgePct'] = (df['TyreLife'] / df['compound_max_life']).clip(0,1).round(3)
print(df[['Compound', 'TyreLife', 'compound_max_life', 'TyreAgePct']].head(10))

  Compound  TyreLife  compound_max_life  TyreAgePct
0     SOFT       1.0               25.0        0.04
1     SOFT       4.0               25.0        0.16
2     SOFT       4.0               25.0        0.16
3     SOFT       3.0               25.0        0.12
4     SOFT       1.0               25.0        0.04
5     SOFT       4.0               25.0        0.16
6     SOFT       4.0               25.0        0.16
7     SOFT       4.0               25.0        0.16
8     SOFT       4.0               25.0        0.16
9     SOFT       1.0               25.0        0.04


In [34]:
df['LapTimeSec'] = df['LapTime'].dt.total_seconds() 

df['BaseLapTime'] = (
    df.groupby(['Year' , 'RoundNumber', 'SessionType', 'Driver'])['LapTimeSec'].transform('first')
) 

df['DegradationRate'] = (df['LapTimeSec'] - df['BaseLapTime']).clip(lower=0).round(3) 

print(df[['Driver', 'Stint', 'TyreLife', 'LapTimeSec', 'BaseLapTime', 'DegradationRate']].head(10))


  Driver  Stint  TyreLife  LapTimeSec  BaseLapTime  DegradationRate
0    LEC    1.0       1.0      99.070       99.070              0.0
1    VER    1.0       4.0     100.236      100.236              0.0
2    SAI    1.0       4.0     101.006      101.006              0.0
3    HAM    1.0       3.0     101.555      101.555              0.0
4    MAG    1.0       1.0     102.333      102.333              0.0
5    PER    1.0       4.0     102.993      102.993              0.0
6    RUS    1.0       4.0     103.445      103.445              0.0
7    GAS    1.0       4.0     104.270      104.270              0.0
8    ALO    1.0       4.0     105.174      105.174              0.0
9    OCO    1.0       1.0     106.170      106.170              0.0


In [35]:
mask = (df['Driver'] == 'VER') & (df['Stint'] == 1) & (df['Year'] == 2024) & (df['RoundNumber'] == 1)

print(df[mask][['Driver', 'TyreLife', 'LapTimeSec', 'BaseLapTime', 'DegradationRate']].head(20))

      Driver  TyreLife  LapTimeSec  BaseLapTime  DegradationRate
44748    VER       4.0      97.284       97.284            0.000
44768    VER       5.0      96.296       97.284            0.000
44787    VER       6.0      96.753       97.284            0.000
44807    VER       7.0      96.647       97.284            0.000
44827    VER       8.0      97.173       97.284            0.000
44847    VER       9.0      97.092       97.284            0.000
44867    VER      10.0      97.038       97.284            0.000
44887    VER      11.0      97.024       97.284            0.000
44907    VER      12.0      97.229       97.284            0.000
44927    VER      13.0      96.960       97.284            0.000
44946    VER      14.0      97.085       97.284            0.000
44965    VER      15.0      97.045       97.284            0.000
44984    VER      16.0      97.030       97.284            0.000
45004    VER      17.0      97.028       97.284            0.000
45024    VER      18.0   

In [36]:
df = df.sort_values(['Year', 'RoundNumber', 'Driver', 'LapNumber']).reset_index(drop=True)
df['HasPitStop'] = df['PitInTime'].notna().astype(int) 

df['WillPitNextLap'] = (
    df.groupby(['Year', 'RoundNumber', 'Driver'])['HasPitStop']
    .shift(-1).fillna(0).astype(int)
) 

print("Distribution:")
print(df['WillPitNextLap'].value_counts())
print(f"\nPit %: {df['WillPitNextLap'].mean()*100:.1f}%") 

mask = (df['Driver'] == 'VER') & (df['Year'] == 2024) & (df['RoundNumber'] == 1)
print(df[mask][['LapNumber', 'TyreLife', 'DegradationRate', 
                'HasPitStop', 'WillPitNextLap']].head(25))

Distribution:
WillPitNextLap
0    92098
1     2679
Name: count, dtype: int64

Pit %: 2.8%
       LapNumber  TyreLife  DegradationRate  HasPitStop  WillPitNextLap
45760        1.0       4.0            0.000           0               0
45761        2.0       5.0            0.000           0               0
45762        3.0       6.0            0.000           0               0
45763        4.0       7.0            0.000           0               0
45764        5.0       8.0            0.000           0               0
45765        6.0       9.0            0.000           0               0
45766        7.0      10.0            0.000           0               0
45767        8.0      11.0            0.000           0               0
45768        9.0      12.0            0.000           0               0
45769       10.0      13.0            0.000           0               0
45770       11.0      14.0            0.000           0               0
45771       12.0      15.0            0.000   

In [37]:
df.loc[df['TyreLife'] == 1, 'DegradationRate'] = 0.0

mask = (df['Driver'] == 'VER') & (df['Year'] == 2024) & (df['RoundNumber'] == 1)
print(df[mask][['LapNumber', 'TyreLife', 'DegradationRate', 'HasPitStop']].head(25))

       LapNumber  TyreLife  DegradationRate  HasPitStop
45760        1.0       4.0            0.000           0
45761        2.0       5.0            0.000           0
45762        3.0       6.0            0.000           0
45763        4.0       7.0            0.000           0
45764        5.0       8.0            0.000           0
45765        6.0       9.0            0.000           0
45766        7.0      10.0            0.000           0
45767        8.0      11.0            0.000           0
45768        9.0      12.0            0.000           0
45769       10.0      13.0            0.000           0
45770       11.0      14.0            0.000           0
45771       12.0      15.0            0.000           0
45772       13.0      16.0            0.000           0
45773       14.0      17.0            0.000           0
45774       15.0      18.0            0.000           0
45775       16.0      19.0            0.000           0
45776       17.0      20.0            2.612     

In [38]:
df['BaseLapTime'] = (
    df.groupby(['Year', 'RoundNumber', 'Driver', 'Stint'])
    ['LapTimeSec'].transform('min')
)

df['DegradationRate'] = (df['LapTimeSec'] - df['BaseLapTime']).clip(lower=0).round(3)
df.loc[df['TyreLife'] == 1, 'DegradationRate'] = 0.0

mask = (df['Driver'] == 'VER') & (df['Year'] == 2024) & (df['RoundNumber'] == 1)
print(df[mask][['LapNumber', 'TyreLife', 'LapTimeSec', 
                'BaseLapTime', 'DegradationRate']].head(25))

       LapNumber  TyreLife  LapTimeSec  BaseLapTime  DegradationRate
45760        1.0       4.0      97.284       96.296            0.988
45761        2.0       5.0      96.296       96.296            0.000
45762        3.0       6.0      96.753       96.296            0.457
45763        4.0       7.0      96.647       96.296            0.351
45764        5.0       8.0      97.173       96.296            0.877
45765        6.0       9.0      97.092       96.296            0.796
45766        7.0      10.0      97.038       96.296            0.742
45767        8.0      11.0      97.024       96.296            0.728
45768        9.0      12.0      97.229       96.296            0.933
45769       10.0      13.0      96.960       96.296            0.664
45770       11.0      14.0      97.085       96.296            0.789
45771       12.0      15.0      97.045       96.296            0.749
45772       13.0      16.0      97.030       96.296            0.734
45773       14.0      17.0      97

In [39]:
total_laps = df.groupby(['Year','RoundNumber'])['LapNumber'].transform('max') 
df['LapPct'] = (df['LapNumber'] / total_laps).round(3) 

COMPOUND_ENCODING = {
    'SOFT' : 0 , 'MEDIUM' :1 , 'HARD' : 2 , 'INTERMEDIATE' : 3 , 'WET' : 4
}
df['CompoundEncoded'] = df['Compound'].map(COMPOUND_ENCODING) 

print(df[['LapNumber', 'LapPct', 'Compound', 'CompoundEncoded']].head(10))

   LapNumber  LapPct Compound  CompoundEncoded
0        1.0   0.018     SOFT              0.0
1        2.0   0.035     SOFT              0.0
2        3.0   0.053     SOFT              0.0
3        4.0   0.070     SOFT              0.0
4        5.0   0.088     SOFT              0.0
5        6.0   0.105     SOFT              0.0
6        7.0   0.123     SOFT              0.0
7        8.0   0.140     SOFT              0.0
8        9.0   0.158     SOFT              0.0
9       10.0   0.175     SOFT              0.0


In [40]:
gold_cols = [
    # Context
    'Year', 'RoundNumber', 'Driver', 'Team',
    'LapNumber', 'LapPct',

    # Tyre Features
    'Compound', 'CompoundEncoded',
    'TyreLife', 'TyreAgePct',
    'DegradationRate',

    # Weather
    'TrackTemp', 'AirTemp', 'Humidity', 'WindSpeed', 'Rainfall',

    # Target
    'WillPitNextLap'
]

gold_df = df[gold_cols].copy()

gold_df.to_parquet(OUTPUT_DIR / "f1_gold.parquet", 
                index=False, compression='snappy') 

print(f"Gold saved : {len(gold_df):,} rows") 
print(f"Features : {gold_df.shape[1]:,} columns") 
print(f"\n Null Check") 
print(gold_df.isnull().sum()[gold_df.isnull().sum() > 0])

Gold saved : 94,777 rows
Features : 17 columns

 Null Check
CompoundEncoded    66
TyreAgePct         66
dtype: int64


In [41]:
print(df[df['CompoundEncoded'].isna()]['Compound'].value_counts())

Compound
None    66
Name: count, dtype: int64


In [42]:
df = df[df['Compound'] != 'None']

df['CompoundEncoded'] = df['Compound'].map(COMPOUND_ENCODING)
df['TyreAgePct'] = (df['TyreLife'] / df['compound_max_life']).clip(0, 1).round(3)

print(f"Rows after fix: {len(df):,}")
print(df['Compound'].value_counts())

Rows after fix: 94,711
Compound
HARD            42130
MEDIUM          35240
SOFT            11899
INTERMEDIATE     5097
WET               345
Name: count, dtype: int64


In [43]:
gold_cols = [
    'Year', 'RoundNumber', 'Driver', 'Team',
    'LapNumber', 'LapPct',
    'Compound', 'CompoundEncoded',
    'TyreLife', 'TyreAgePct',
    'DegradationRate',
    'TrackTemp', 'AirTemp', 'Humidity', 'WindSpeed', 'Rainfall',
    'WillPitNextLap'
]

gold_df = df[gold_cols].copy()

gold_df.to_parquet(OUTPUT_DIR / "f1_gold.parquet",
                index=False, compression='snappy')

print(f"Gold saved: {len(gold_df):,} rows")
print(f"Features: {gold_df.shape[1]}")
print(f"\nNull check:")
print(gold_df.isnull().sum()[gold_df.isnull().sum() > 0])

Gold saved: 94,711 rows
Features: 17

Null check:
Series([], dtype: int64)


In [44]:
df['BaseLapTime'] = ( 
    df.groupby(['Year', 'RoundNumber', 'Driver', 'Stint'])['LapTimeSec'].transform(lambda x : x.head(3).median()) 
) 

df['DegradationRate'] = (
    df['LapTimeSec'] - df['BaseLapTime']
).clip(lower=0).round(3) 

df.loc[df['TyreLife'] == 1, 'DegradationRate'] = 0.0 

In [45]:
from scipy import stats 

def calc_slope(group):
    if len(group)<3 : 
        group['DegradationSlope'] = 0.0 
        return group 

    slope , _ , _ , _ , _ = stats.linregress(group['TyreLife'] , group['LapTimeSec']) 

    group['DegradationSlope'] = max(0 , round(slope , 4)) 
    return group 

df = (
    df.groupby(['Year' , 'RoundNumber' , 'Driver' , 'Stint'] , group_keys = False)
    .apply(calc_slope)
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_17336\3185150900.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(calc_slope)


In [46]:
print(df[['Driver', 'Stint', 'TyreLife', 'LapTimeSec', 'DegradationRate', 'DegradationSlope']].head(10))

  Driver  Stint  TyreLife  LapTimeSec  DegradationRate  DegradationSlope
0    ALB    1.0       1.0     107.636            0.000             0.046
1    ALB    1.0       2.0     100.548            0.000             0.046
2    ALB    1.0       3.0     100.664            0.000             0.046
3    ALB    1.0       4.0     101.126            0.462             0.046
4    ALB    1.0       5.0     102.303            1.639             0.046
5    ALB    1.0       6.0     101.708            1.044             0.046
6    ALB    1.0       7.0     101.561            0.897             0.046
7    ALB    1.0       8.0     103.097            2.433             0.046
8    ALB    1.0       9.0     102.149            1.485             0.046
9    ALB    1.0      10.0     103.500            2.836             0.046


In [47]:
gold_cols = [
    'Year', 'RoundNumber', 'Driver', 'Team',
    'LapNumber', 'LapPct',
    'Compound', 'CompoundEncoded',
    'TyreLife', 'TyreAgePct',
    'DegradationRate', 'DegradationSlope',
    'TrackTemp', 'AirTemp', 'Humidity', 'WindSpeed', 'Rainfall',
    'WillPitNextLap' , 'Position' 
]

gold_df = df[gold_cols].copy()

gold_df.to_parquet(OUTPUT_DIR / "f1_gold.parquet",
                index=False, compression='snappy')

print(f"Gold saved: {len(gold_df):,} rows")
print(f"Features: {gold_df.shape[1]}")
print(f"\nNull check:")
print(gold_df.isnull().sum()[gold_df.isnull().sum() > 0])

Gold saved: 94,711 rows
Features: 19

Null check:
Series([], dtype: int64)


In [48]:
print(df['Position'].value_counts().head())
print(f"Position nulls: {df['Position'].isna().sum()}")

Position
6.0    5179
1.0    5174
7.0    5169
5.0    5157
3.0    5157
Name: count, dtype: int64
Position nulls: 0


In [49]:
OUTPUT_DIR = Path(r"C:\f1 new\data")

sc_df = pd.read_parquet(OUTPUT_DIR / "f1_race_control.parquet")
print(f"Silver: {len(df):,} rows")

Silver: 94,711 rows


In [50]:
sc_df['Lap'] = sc_df['Lap'].fillna(0).astype(int)

sc_laps = (
    sc_df.groupby(['Year', 'RoundNumber'])['Lap']
    .apply(set)  
    .reset_index()
)
sc_laps.columns = ['Year', 'RoundNumber', 'SCLaps']

# ③ Merge
df = df.merge(sc_laps, on=['Year', 'RoundNumber'], how='left')
df['SCLaps'] = df['SCLaps'].apply(lambda x: x if isinstance(x, set) else set())

# ④ Feature
def sc_in_next3(row):
    next_laps = {int(row['LapNumber']) + i for i in range(1, 4)}
    return int(bool(next_laps & row['SCLaps']))

df['SCInNext3Laps'] = df.apply(sc_in_next3, axis=1)

print(df['SCInNext3Laps'].value_counts())

SCInNext3Laps
1    72288
0    22423
Name: count, dtype: int64


In [51]:
sample = df[
    (df['Year'] == 2022) & 
    (df['RoundNumber'] == 1)
][['LapNumber', 'SCLaps', 'SCInNext3Laps']].head(5)

print(sample)
print()

print(sc_laps[
    (sc_laps['Year'] == 2022) & 
    (sc_laps['RoundNumber'] == 1)
])

   LapNumber                                             SCLaps  SCInNext3Laps
0        1.0  {1, 2, 3, 4, 6, 8, 9, 15, 26, 29, 30, 31, 33, ...              1
1        2.0  {1, 2, 3, 4, 6, 8, 9, 15, 26, 29, 30, 31, 33, ...              1
2        3.0  {1, 2, 3, 4, 6, 8, 9, 15, 26, 29, 30, 31, 33, ...              1
3        4.0  {1, 2, 3, 4, 6, 8, 9, 15, 26, 29, 30, 31, 33, ...              1
4        5.0  {1, 2, 3, 4, 6, 8, 9, 15, 26, 29, 30, 31, 33, ...              1

   Year  RoundNumber                                             SCLaps
0  2022            1  {1, 2, 3, 4, 6, 8, 9, 15, 26, 29, 30, 31, 33, ...


In [52]:
print(sc_df[
    (sc_df['Year'] == 2022) & 
    (sc_df['RoundNumber'] == 1)
][['Message', 'Lap']].to_string())

                                                                                                                                       Message  Lap
0                                                                                                                  GREEN LIGHT - PIT EXIT OPEN    1
1                                                                                                                              PIT EXIT CLOSED    1
2                                                                                                               RISK OF RAIN FOR F1 RACE IS 0%    1
3                                                                                                                                 DRS DISABLED    1
4                                                                                                                  GREEN LIGHT - PIT EXIT OPEN    1
5                                                             TURN 6 INCIDENT INVOLVING CARS 31 (OCO) AND 47 (MS

In [53]:
sc_only = sc_df[sc_df['Message'].str.contains(
    'SAFETY CAR DEPLOYED|VIRTUAL SAFETY CAR DEPLOYED', 
    na=False
)].copy()

print(f"SC Events: {len(sc_only)}")
print(sc_only[['Year', 'RoundNumber', 'Message', 'Lap']].head(10))

sc_only['Lap'] = sc_only['Lap'].fillna(0).astype(int)

sc_laps = (
    sc_only.groupby(['Year', 'RoundNumber'])['Lap']
    .apply(set)
    .reset_index()
)
sc_laps.columns = ['Year', 'RoundNumber', 'SCLaps']

df = df.drop(columns=['SCLaps', 'SCInNext3Laps'], errors='ignore')
df = df.merge(sc_laps, on=['Year', 'RoundNumber'], how='left')
df['SCLaps'] = df['SCLaps'].apply(lambda x: x if isinstance(x, set) else set())

def sc_in_next3(row):
    next_laps = {int(row['LapNumber']) + i for i in range(1, 4)}
    return int(bool(next_laps & row['SCLaps']))

df['SCInNext3Laps'] = df.apply(sc_in_next3, axis=1)

print(df['SCInNext3Laps'].value_counts())

SC Events: 121
     Year  RoundNumber                      Message  Lap
47   2022            1  VIRTUAL SAFETY CAR DEPLOYED   46
48   2022            1          SAFETY CAR DEPLOYED   46
97   2022            2  VIRTUAL SAFETY CAR DEPLOYED   16
98   2022            2          SAFETY CAR DEPLOYED   16
119  2022            2  VIRTUAL SAFETY CAR DEPLOYED   38
156  2022            3  VIRTUAL SAFETY CAR DEPLOYED    3
157  2022            3          SAFETY CAR DEPLOYED    3
173  2022            3          SAFETY CAR DEPLOYED   23
181  2022            3  VIRTUAL SAFETY CAR DEPLOYED   39
223  2022            4          SAFETY CAR DEPLOYED    1
SCInNext3Laps
0    90276
1     4435
Name: count, dtype: int64


In [54]:
df.columns

Index(['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint',
       'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
       'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
       'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest',
       'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime',
       'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason',
       'FastF1Generated', 'IsAccurate', 'AirTemp', 'TrackTemp', 'Humidity',
       'WindSpeed', 'Rainfall', 'Year', 'RoundNumber', 'Country', 'EventName',
       'SessionType', 'LapTimeSec', 'compound_max_life', 'TyreAgePct',
       'BaseLapTime', 'DegradationRate', 'HasPitStop', 'WillPitNextLap',
       'LapPct', 'CompoundEncoded', 'DegradationSlope', 'SCLaps',
       'SCInNext3Laps'],
      dtype='object')

In [55]:
gold_cols = [
    'Year', 'RoundNumber', 'Driver', 'Team',
    'LapNumber', 'LapPct',
    'Compound', 'CompoundEncoded',
    'TyreLife', 'TyreAgePct',
    'DegradationRate', 'DegradationSlope',
    'Position',
    'SCInNext3Laps',
    'TrackTemp', 'AirTemp', 'Humidity', 'WindSpeed', 'Rainfall',
    'WillPitNextLap'
] 

gold_df = df[gold_cols].copy() 
gold_df.to_parquet(OUTPUT_DIR/ "f1_gold.parquet" , index=False, compression='snappy') 

print(f"Gold saved: {len(gold_df):,} rows")
print(f"Features: {gold_df.shape[1]}")
print(f"\nNull check:")
print(gold_df.isnull().sum()[gold_df.isnull().sum() > 0])

Gold saved: 94,711 rows
Features: 20

Null check:
Series([], dtype: int64)
